# Stock Prediction with PyTorch - Latest WSB Stocks

This notebook trains a lightweight PyTorch LSTM on the latest WallStreetBets stocks using current Yahoo Finance data.

It now:
- refreshes the latest WSB ranking,
- defaults to the top 2 current tickers,
- trains a compact LSTM per ticker on close prices,
- reports RMSE and saves charts/results under `outputs/latest_wsb_analysis/`.

## Step 1 - Setup

In [1]:
from pathlib import Path
import json
import sys
from typing import cast

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from torch import nn

try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()

if not (ROOT / "src").exists():
    ROOT = Path.cwd()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.wsb_latest.pipeline import fetch_price_history, run_pipeline

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cpu")
plt.rcParams["figure.figsize"] = (12, 6)
print(f"Project root: {ROOT}")
print(f"Torch device: {device}")

Project root: /home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets
Torch device: cpu


## Step 2 - Configuration

In [2]:
OUTPUT_DIR = ROOT / "outputs" / "latest_wsb_analysis"
CHARTS_DIR = OUTPUT_DIR / "charts"
PRICES_DIR = OUTPUT_DIR / "prices"
PRICE_PERIOD = "2y"
TOP_K = 2
import os
REFRESH_WSB_DATA = os.getenv("WSB_REFRESH_DATA", "0") != "0"
MANUAL_TICKERS = None
WINDOW_SIZE = 20
EPOCHS = 25
LEARNING_RATE = 0.01

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
PRICES_DIR.mkdir(parents=True, exist_ok=True)

## Step 3 - Refresh or load the latest WSB ranking

In [3]:
summary_path = OUTPUT_DIR / "top10_wsb_stocks.csv"

if REFRESH_WSB_DATA or (not summary_path.exists()):
    result = run_pipeline(top_n=10, per_feed=100, price_period="1y", output_dir=OUTPUT_DIR)
    print(json.dumps(result, indent=2))

summary_df = cast(pd.DataFrame, pd.read_csv(str(summary_path)))
if MANUAL_TICKERS is not None:
    selected_tickers = [ticker.upper() for ticker in MANUAL_TICKERS]
else:
    selected_tickers = summary_df["ticker"].head(TOP_K).tolist()

print("Selected tickers:", selected_tickers)

Selected tickers: ['AMD', 'NVDA']


## Step 4 - Prepare PyTorch sequence helpers

In [4]:
class PriceLSTM(nn.Module):
    def __init__(self, hidden_size=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        return self.fc(out)


def create_sequences(values, window_size):
    X, y = [], []
    for idx in range(window_size, len(values)):
        X.append(values[idx - window_size: idx])
        y.append(values[idx])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def fit_lstm_to_close(close_series, window_size=20, epochs=25, learning_rate=0.01):
    data = close_series.dropna().to_frame(name="Close")
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(data)
    X, y = create_sequences(scaled, window_size)
    if len(X) < 20:
        raise ValueError("Not enough price rows to train the LSTM.")

    split = max(int(len(X) * 0.8), 1)
    split = min(split, len(X) - 1)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)

    model = PriceLSTM().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.MSELoss()
    losses = []

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        pred = model(X_train_t)
        loss = loss_fn(pred, y_train_t)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.item()))

    model.eval()
    with torch.no_grad():
        test_pred_scaled = model(X_test_t).cpu().numpy()

    actual = scaler.inverse_transform(y_test)
    predicted = scaler.inverse_transform(test_pred_scaled)
    rmse = float(np.sqrt(mean_squared_error(actual, predicted)))
    result_index = close_series.index[-len(actual):]
    results_df = pd.DataFrame({"Actual": actual.ravel(), "Predicted": predicted.ravel()}, index=result_index)
    return model, losses, rmse, results_df

## Step 5 - Train a compact LSTM for the selected latest WSB tickers

In [5]:
model_outputs = {}
summary_rows = []

for ticker in selected_tickers:
    history_df = fetch_price_history(ticker, PRICE_PERIOD)
    close_series = history_df["Close"].copy()
    model, losses, rmse, results_df = fit_lstm_to_close(
        close_series,
        window_size=WINDOW_SIZE,
        epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
    )

    result_csv = PRICES_DIR / f"{ticker.lower()}_pytorch_lstm_results.csv"
    chart_png = CHARTS_DIR / f"{ticker.lower()}_pytorch_lstm.png"
    results_df.to_csv(result_csv)

    fig, axes = plt.subplots(2, 1, figsize=(12, 9))
    axes[0].plot(losses, color="#1f77b4")
    axes[0].set_title(f"{ticker} training loss")
    axes[1].plot(results_df.index, results_df["Actual"], label="Actual", color="#2ca02c")
    axes[1].plot(results_df.index, results_df["Predicted"], label="Predicted", color="#d62728")
    axes[1].set_title(f"{ticker} predicted vs actual close")
    axes[1].legend(loc="upper left")
    plt.tight_layout()
    plt.savefig(chart_png, dpi=180)
    plt.show()

    model_outputs[ticker] = {
        "losses": losses,
        "rmse": rmse,
        "results": results_df,
        "result_csv": result_csv,
        "chart_png": chart_png,
    }
    summary_rows.append({
        "ticker": ticker,
        "rows": int(len(close_series.dropna())),
        "rmse": rmse,
        "result_csv": str(result_csv),
        "chart_png": str(chart_png),
    })

lstm_summary_df = pd.DataFrame(summary_rows).sort_values("ticker").reset_index(drop=True)
lstm_summary_path = OUTPUT_DIR / "pytorch_lstm_summary.csv"
lstm_summary_df.to_csv(lstm_summary_path, index=False)
lstm_summary_df

,ticker,rows,rmse,result_csv,chart_png
0,AMD,501,67.547911,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...
1,NVDA,501,6.829989,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...


## Step 6 - Review the latest prediction outputs

In [6]:
for ticker in selected_tickers:
    print(f"\n===== {ticker} LSTM output =====")
    print(model_outputs[ticker]["results"].tail(10).to_string())


===== AMD LSTM output =====
                Actual   Predicted
Date                              
2026-04-24  347.809998  197.537460
2026-04-27  334.630005  203.894791
2026-04-28  323.209991  205.970016
2026-04-29  337.109985  205.601196
2026-04-30  354.489990  206.959045
2026-05-01  360.540039  209.879669
2026-05-04  341.540009  212.331665
2026-05-05  355.260040  211.366226
2026-05-06  421.390015  212.392532
2026-05-07  408.459991  221.091370

===== NVDA LSTM output =====
                Actual   Predicted
Date                              
2026-04-24  208.270004  198.817276
2026-04-27  216.610001  201.445190
2026-04-28  213.169998  205.061447
2026-04-29  209.250000  207.447388
2026-04-30  199.570007  208.778961
2026-05-01  198.449997  208.328568
2026-05-04  198.479996  207.894073
2026-05-05  196.500000  207.566788
2026-05-06  207.830002  206.961349
2026-05-07  211.500000  208.359283


## Step 7 - Final notebook summary

In [7]:
notebook_summary = {
    "selected_tickers": selected_tickers,
    "window_size": WINDOW_SIZE,
    "epochs": EPOCHS,
    "summary_csv": str(lstm_summary_path.resolve()),
    "output_dir": str(OUTPUT_DIR.resolve()),
}
print(json.dumps(notebook_summary, indent=2))

{
  "selected_tickers": [
    "AMD",
    "NVDA"
  ],
  "window_size": 20,
  "epochs": 25,
  "summary_csv": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis/pytorch_lstm_summary.csv",
  "output_dir": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis"
}
